In [1]:
# SPDX-License-Identifier: MIT
# Copyright (c) 2025 Hammerheads Engineers sp. z o.o.
# Author: Aleksander Stanik
import sys
import os
import time
import yaml
import json
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

current_dir = os.getcwd()
repo_root = os.path.abspath(os.path.join(current_dir, '..'))

if repo_root not in sys.path:
    sys.path.append(repo_root)


import spx_python
# Initialize HTTP-based SPX client wrapper pointing to local SPX server
product_key = os.environ['SPX_PRODUCT_KEY']
wrapper = spx_python.init(address='http://localhost:8000',
                                product_key=product_key)

In [2]:
pid_system_yaml = """
models:
    TemperatureSensor:
        attributes:
            temperature: 25.0
            heating_power:
                default: 0.0
                hooks:
                    on_set:
                        - refresh_model
        actions:
            - function: $in(temperature)
              call: $in(temperature) + 0.5 * (0.6 * $in(heating_power) - 0.02 * ($in(temperature) - 25))

    PowerSupply:
        attributes:
            power: 0.0
            power_on: true
        conditions:
            - if: "not $in(power_on)"
              actions:
                - set: "$in(power)"
                  value: 0.0

    PIDController:
        attributes:
            setpoint: 140.0
            output: 0.0
            input: 25.0
        actions:
            - pid: $in(output)
              setpoint: 140.0
              kp: 2.5
              ki: 0.01
              kd: 0.01
              input: '$in(input)'
instances:
    - sensor: TemperatureSensor
    - supply: PowerSupply
    - controller: PIDController

connections:
    sensor_to_controller:
        from: $out(sensor.temperature)
        to: $in(controller.input)
    controller_to_supply:
        from: $out(controller.output)
        to: $in(supply.power)
    supply_to_sensor:
        from: $out(supply.power)
        to: $in(sensor.heating_power)
"""

# Parse YAML and build the model
data = yaml.safe_load(pid_system_yaml)
print("Parsed YAML data:", data)

wrapper['models'] = data['models']
wrapper['instances'] = data['instances']
wrapper['connections'] = data.get('connections', {})

system = wrapper

print("Available models:", wrapper["models"].keys())
print("Available instances:", wrapper["instances"].keys())

Parsed YAML data: {'models': {'TemperatureSensor': {'attributes': {'temperature': 25.0, 'heating_power': {'default': 0.0, 'hooks': {'on_set': ['refresh_model']}}}, 'actions': [{'function': '$in(temperature)', 'call': '$in(temperature) + 0.5 * (0.6 * $in(heating_power) - 0.02 * ($in(temperature) - 25))'}]}, 'PowerSupply': {'attributes': {'power': 0.0, 'power_on': True}, 'conditions': [{'if': 'not $in(power_on)', 'actions': [{'set': '$in(power)', 'value': 0.0}]}]}, 'PIDController': {'attributes': {'setpoint': 140.0, 'output': 0.0, 'input': 25.0}, 'actions': [{'pid': '$in(output)', 'setpoint': 140.0, 'kp': 2.5, 'ki': 0.01, 'kd': 0.01, 'input': '$in(input)'}]}}, 'instances': [{'sensor': 'TemperatureSensor'}, {'supply': 'PowerSupply'}, {'controller': 'PIDController'}], 'connections': {'sensor_to_controller': {'from': '$out(sensor.temperature)', 'to': '$in(controller.input)'}, 'controller_to_supply': {'from': '$out(controller.output)', 'to': '$in(supply.power)'}, 'supply_to_sensor': {'from':

In [4]:
time_steps = 300
times = np.linspace(0, 60, time_steps)
temperature_values = []
power_values = []
power_on_values = []
pid_input_values = []
pid_output_values = []

system["polling"].enabled = False
system["instances"]['sensor']['attributes']['temperature'].internal_value = 25.0

system.reset()
system.prepare()

for t in times:
    # for instance in system["instances"].values():
    #   instance["timer"]["time"] = t
    system["timer"].time = t

    system.run()

    # Getting values for plots
    temperature_sensor = system["instances"]['sensor']["attributes"]
    pid_controller = system["instances"]['controller']["attributes"]
    power_supply = system["instances"]['supply']["attributes"]

    temperature_values.append(temperature_sensor['temperature'].internal_value)
    power_values.append(power_supply['power'].internal_value)
    power_on_values.append(power_supply['power_on'].internal_value)
    pid_input_values.append(pid_controller['input'].internal_value)
    pid_output_values.append(pid_controller['output'].internal_value)
    print(f"Time: {t:.2f}s, Temperature: {temperature_values[-1]:.2f}°C")

# Create figure
fig = go.Figure()

# Temperature plot on primary y-axis (y1)
fig.add_trace(go.Scatter(
    x=times,
    y=temperature_values,
    mode='lines',
    name='Temperature (°C)',
    yaxis='y1'
))
fig.add_trace(go.Scatter(
    x=[0, times[-1]],
    y=[150, 150],
    mode='lines',
    name='Shutdown threshold (°C)',
    line=dict(dash='dash', color='red'),
    yaxis='y1'
))
fig.add_trace(go.Scatter(
    x=[0, times[-1]],
    y=[120, 120],
    mode='lines',
    name='Restart threshold (°C)',
    line=dict(dash='dash', color='green'),
    yaxis='y1'
))

# Power plot on secondary y-axis (y2)
fig.add_trace(go.Scatter(
    x=times,
    y=power_values,
    mode='lines',
    name='Power Supply Power (W)',
    yaxis='y2'
))

# PowerOn plot on tertiary y-axis (y3)
power_on_values_numeric = [1 if val else 0 for val in power_on_values]
fig.add_trace(go.Scatter(
    x=times,
    y=power_on_values_numeric,
    mode='lines',
    name='PowerOn State',
    yaxis='y3'
))

# Update layout with three y-axes and units
fig.update_layout(
    title='Temperature Regulation System Simulation',
    xaxis=dict(title='Time (s)'),
    yaxis=dict(
        title='Temperature (°C)',
        side='left',
        showgrid=True,
    ),
    yaxis2=dict(
        title='Power (W)',
        side='left',
        overlaying='y',
        position=0.0,
        showgrid=False,
    ),
    yaxis3=dict(
        title='PowerOn State',
        side='right',
        overlaying='y',
        position=1.0,
        showgrid=False,
    ),
    legend=dict(
        x=1.15,
        y=1,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255,255,255,0)',
        bordercolor='black',
        borderwidth=1
    ),
    margin=dict(r=200)  # Increase right margin to make space for legend and extra axis
)

fig.show()

Time: 0.00s, Temperature: 130.00°C
Time: 0.20s, Temperature: 318.11°C
Time: 0.40s, Temperature: 483.92°C
Time: 0.60s, Temperature: 489.71°C
Time: 0.80s, Temperature: 214.63°C
Time: 1.00s, Temperature: -302.44°C
Time: 1.20s, Temperature: -817.86°C
Time: 1.40s, Temperature: -912.47°C
Time: 1.61s, Temperature: -233.48°C
Time: 1.81s, Temperature: 1201.27°C
Time: 2.01s, Temperature: 2748.68°C
Time: 2.21s, Temperature: 3251.89°C
Time: 2.41s, Temperature: 1603.73°C
Time: 2.61s, Temperature: -2321.13°C
Time: 2.81s, Temperature: -6918.95°C
Time: 3.01s, Temperature: -8965.38°C
Time: 3.21s, Temperature: -5113.23°C
Time: 3.41s, Temperature: 5524.50°C
Time: 3.61s, Temperature: 19004.84°C
Time: 3.81s, Temperature: 26467.59°C
Time: 4.01s, Temperature: 17905.02°C
Time: 4.21s, Temperature: -10606.56°C
Time: 4.41s, Temperature: -49688.92°C
Time: 4.62s, Temperature: -75213.92°C
Time: 4.82s, Temperature: -57677.42°C
Time: 5.02s, Temperature: 17840.53°C
Time: 5.22s, Temperature: 129951.72°C
Time: 5.42s, Te